# Clean a fun run's results

The timing desk of a 5 km fun run exports its results as a CSV, and the export has the
usual faults: club names typed several ways, an age written as `?`, a runner who did not
finish, a repeated row and a runner scanned twice at the finish line. You will clean it
into one row per finisher and then summarize it by club.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import io
import numpy as np
import pandas as pd

# The timing desk's export of a 5 km fun run. A runner scanned twice at the
# finish line has two rows, and the later scan is the corrected one.
export = """bib,name,club,age,time
101,Mia Chen,harbor rc,15,24:10
102,Leo Park, Harbor RC ,16,22:45
103,Ivy Moss,valley striders,?,26:30
104,Sam Ortiz,VALLEY STRIDERS,17,21:05
102,Leo Park, Harbor RC ,16,22:45
105,Ada Kim,harbor rc,,25:40
106,Tom Reyes,Hill Runners,15,DNF
107,Zoe Hart,valley striders ,16,23:55
104,Sam Ortiz,VALLEY STRIDERS,17,20:58
108,Eli Brooks,hill runners,18,27:20
"""
print(pd.read_csv(io.StringIO(export)))

## Part 1 (coding task): Clean the export

Write `clean_results(text)`, which reads the CSV text and returns a DataFrame with the
columns `bib`, `name`, `club`, `age` and `seconds`, where

- `club` has no spaces at either end and is in lower case, so each club has one spelling.
- `age` is numeric, and an age that is not a number becomes NaN.
- `seconds` is the finish time in seconds, from the `"mm:ss"` text in `time` (24:10 is
  $24 \cdot 60 + 10 = 1450$), and the `time` column is gone.
- a runner who did not finish (`DNF`) has no row.
- each bib appears once, with the values of its **last** row in the file.
- the index runs 0, 1, 2, ... again.

A hint for the times: `s.str.split(":", expand=True)` splits every entry at the colon and
returns a DataFrame with one column per piece, and `pd.to_numeric(..., errors="coerce")`
turns a piece that is not a number, or is missing, into NaN.

In [ ]:
def clean_results(text):
    """The fun run results with one row per finisher, cleaned as described above."""
    runs = pd.read_csv(io.StringIO(text))
    # TODO
    raise NotImplementedError

In [ ]:
results = clean_results(export)
print(results)
assert isinstance(results, pd.DataFrame), "return a DataFrame"
assert sorted(results.columns) == sorted(["bib", "name", "club", "age", "seconds"]), \
    f"the columns should be bib, name, club, age and seconds, not {list(results.columns)}"
assert list(results.index) == list(range(len(results))), "reset the index so it runs 0, 1, 2, ..."
assert 106 not in set(results["bib"]), "bib 106 did not finish, so it should have no row"
assert results["bib"].is_unique, "each bib should appear once: compare rows by bib, not by every column"
assert len(results) == 7, f"expected 7 finishers, got {len(results)}"
clubs = sorted(results["club"].unique())
assert clubs == ["harbor rc", "hill runners", "valley striders"], \
    f"each club should have one spelling, stripped and lower case: {clubs}"
by_bib = results.set_index("bib")
assert pd.api.types.is_numeric_dtype(results["age"]), "age should be numeric"
assert by_bib["age"].isna().sum() == 2, "the ages '?' and empty should both be missing"
assert pd.api.types.is_numeric_dtype(results["seconds"]), "seconds should be numeric"
assert by_bib.loc[101, "seconds"] == 1450, f"24:10 is 1450 seconds, got {by_bib.loc[101, 'seconds']}"
assert by_bib.loc[104, "seconds"] != 1265, "bib 104 should keep its later, corrected scan (20:58), not the first"
assert by_bib.loc[104, "seconds"] == 1258, f"bib 104's corrected time 20:58 is 1258 seconds, got {by_bib.loc[104, 'seconds']}"
expected = {101: 1450, 102: 1365, 103: 1590, 104: 1258, 105: 1540, 107: 1435, 108: 1640}
assert by_bib["seconds"].to_dict() == expected, f"some times are wrong: {by_bib['seconds'].to_dict()}"
print("Part 1 passed")

## Part 2 (coding task): One row per club

Write `club_summary(results)`, which takes a table like the one `clean_results` returns
and gives one row per club, indexed by the club name, with the columns

- `runners`: how many finishers the club has, whether or not their age is known.
- `best`: the club's fastest time in seconds.
- `mean_age`: the mean of the ages that are known.

Sort the rows from the fastest club to the slowest, by `best`.

In [ ]:
def club_summary(results):
    """Finishers, best time and mean known age per club, fastest club first."""
    # TODO
    raise NotImplementedError

In [ ]:
# A cleaned table, written out here so this check does not depend on Part 1.
finishers = pd.DataFrame({
    "bib": [101, 102, 103, 104, 105, 107, 108],
    "name": ["Mia Chen", "Leo Park", "Ivy Moss", "Sam Ortiz", "Ada Kim", "Zoe Hart", "Eli Brooks"],
    "club": ["harbor rc", "harbor rc", "valley striders", "valley striders",
             "harbor rc", "valley striders", "hill runners"],
    "age": [15, 16, np.nan, 17, np.nan, 16, 18],
    "seconds": [1450, 1365, 1590, 1258, 1540, 1435, 1640],
})
summary = club_summary(finishers)
print(summary)
for col in ["runners", "best", "mean_age"]:
    assert col in summary.columns, f"the summary has no {col!r} column"
assert set(summary.index) == {"harbor rc", "hill runners", "valley striders"}, \
    "the summary should be indexed by club name, one row per club"
assert list(summary.index) == ["valley striders", "harbor rc", "hill runners"], \
    f"sort by best time, fastest first: {list(summary.index)}"
runners = summary["runners"].to_dict()
assert runners != {"valley striders": 2, "harbor rc": 2, "hill runners": 1}, \
    "runners counts only the known ages: count every finisher, missing age or not (size, not count)"
assert runners == {"valley striders": 3, "harbor rc": 3, "hill runners": 1}, f"runners is wrong: {runners}"
assert summary["best"].to_dict() == {"valley striders": 1258, "harbor rc": 1365, "hill runners": 1640}, \
    f"best should be each club's smallest time: {summary['best'].to_dict()}"
assert np.isclose(summary.loc["harbor rc", "mean_age"], 15.5), \
    f"harbor rc's mean age should average its two known ages, 15 and 16, not {summary.loc['harbor rc', 'mean_age']}"
assert np.allclose(summary["mean_age"].to_numpy(), [16.5, 15.5, 18.0]), f"mean_age is wrong: {summary['mean_age'].to_dict()}"
assert finishers["age"].isna().sum() == 2, "leave the input table unchanged"
print("Part 2 passed")

## Solution

### Part 1: Clean the export

In [ ]:
def clean_results(text):
    """The fun run results with one row per finisher, cleaned as described above."""
    runs = pd.read_csv(io.StringIO(text))
    runs["club"] = runs["club"].str.strip().str.lower()
    runs["age"] = pd.to_numeric(runs["age"], errors="coerce")       # "?" becomes NaN
    pieces = runs["time"].str.split(":", expand=True)
    minutes = pd.to_numeric(pieces[0], errors="coerce")
    secs = pd.to_numeric(pieces[1], errors="coerce")
    runs["seconds"] = minutes * 60 + secs
    runs = runs.dropna(subset=["seconds"])                       # DNF has no time
    runs = runs.drop_duplicates(subset=["bib"], keep="last")     # the corrected scan
    runs = runs.drop(columns=["time"])
    return runs.reset_index(drop=True)

**Why it works.** Each cleaning step works on a whole column at once. `str.split(":", expand=True)`
puts the minutes and the seconds in two columns, and `"DNF"`, which has no colon, gives NaN
minutes and a missing second piece, so its `seconds` is NaN and `dropna` removes the row.
`drop_duplicates(subset=["bib"], keep="last")` compares bibs only: a plain `drop_duplicates()`
removes Leo's exact repeat but keeps both of Sam's rows, because their times differ, and
`keep="first"` would keep his uncorrected 21:05. The club names are cleaned before anything
is grouped, so `"harbor rc"` and `" Harbor RC "` count as one club.

### Part 2: One row per club

In [ ]:
def club_summary(results):
    """Finishers, best time and mean known age per club, fastest club first."""
    summary = results.groupby("club").agg(
        runners=("bib", "size"),
        best=("seconds", "min"),
        mean_age=("age", "mean"),
    )
    return summary.sort_values("best")

**Why it works.** Named aggregation builds the three columns in one pass, each from its own
column and function. `size` counts the rows of a group, so Ivy and Ada are counted although
their ages are missing, while `count` on `age` would give 2 for both large clubs. `mean`
skips NaN, so harbor rc's mean age is $(15 + 16)/2 = 15.5$ over its two known ages. Filling
the gaps with 0 first would give $31/3 \approx 10.3$, a club of ten-year-olds.